# KonkaniVani ASR - Complete Retraining (FIXED)

## What's Fixed:
- ✅ CTC weight increased to 0.8 (was 0.3)
- ✅ Using full 88-hour dataset (was 21h)
- ✅ Periodic testing every 5 epochs
- ✅ Better learning rate and gradient clipping

## Expected Results:
- Epoch 20: Blank prob < 80% (model starts working)
- Epoch 40: Blank prob < 60% (good transcriptions)
- Epoch 100: Blank prob < 40% (production ready)

## Step 1: Setup Environment

In [ ]:
# Install dependencies
!pip install -q torch torchaudio librosa soundfile jiwer pyyaml tensorboard

In [ ]:
import os
import sys
import json
import torch
import torchaudio
from pathlib import Path
import numpy as np
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 2: Check Dataset

In [ ]:
# List available datasets
!ls -lh /kaggle/input/

In [ ]:
# Set paths - UPDATE THIS to match your dataset name
DATA_ROOT = Path('/kaggle/input/konkani-asr-complete-data')

# Check structure
print("Dataset structure:")
!ls -lh {DATA_ROOT}

## Step 3: Extract and Prepare Data

In [ ]:
# Extract if zipped
import zipfile

zip_files = list(DATA_ROOT.glob('*.zip'))
if zip_files:
    print(f"Found {len(zip_files)} zip files. Extracting...")
    for zip_file in zip_files:
        print(f"Extracting {zip_file.name}...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall('/kaggle/working/')
    print("✓ Extraction complete!")
else:
    print("No zip files found, data already extracted")

In [ ]:
# Check extracted data
!ls -lh /kaggle/working/

## Step 4: Prepare Full Dataset (88 hours)

In [ ]:
# Run data preparation script
# This combines KonkaniRawSpeechCorpus (84h) + existing data (21h)
!python /kaggle/working/scripts/prepare_raw_corpus_data.py

In [ ]:
# Verify manifests created
manifest_dir = Path('/kaggle/working/data/konkani-combined/manifests')
if manifest_dir.exists():
    print("✓ Manifests created:")
    for manifest in manifest_dir.glob('*.json'):
        with open(manifest) as f:
            data = json.load(f)
            print(f"  {manifest.name}: {len(data)} samples")
else:
    print("✗ Manifests not found, using existing data")
    manifest_dir = Path('/kaggle/working/data/konkani-asr-v0/splits/manifests')

## Step 5: Configure Training (FIXED SETTINGS)

In [ ]:
# Training configuration with FIXES
import yaml

config = {
    'model': {
        'vocab_size': 200,
        'input_dim': 80,
        'd_model': 256,
        'encoder_layers': 12,
        'decoder_layers': 6,
        'num_heads': 4,
        'conv_kernel_size': 31,
        'dropout': 0.2
    },
    'training': {
        'learning_rate': 0.0003,      # 🔥 Increased from 0.0001
        'weight_decay': 0.0001,
        'grad_clip': 5.0,             # 🔥 Added gradient clipping
        'ctc_weight': 0.8,            # 🔥 CRITICAL FIX: was 0.3
        'batch_size': 2,
        'gradient_accumulation_steps': 4,
        'mixed_precision': True,
        'num_epochs': 100,            # 🔥 More epochs
        'save_every': 5,
        'test_every': 5               # 🔥 Test every 5 epochs
    },
    'data': {
        'train_manifest': str(manifest_dir / 'train.json'),
        'val_manifest': str(manifest_dir / 'val.json'),
        'vocab_file': '/kaggle/working/data/vocab.json',
        'num_workers': 2
    },
    'paths': {
        'checkpoint_dir': '/kaggle/working/checkpoints',
        'log_dir': '/kaggle/working/logs'
    },
    'device': 'cuda'
}

# Save config
os.makedirs('/kaggle/working/config', exist_ok=True)
with open('/kaggle/working/config/training_config_fixed.yaml', 'w') as f:
    yaml.dump(config, f)

print("✓ Training config saved with FIXES:")
print(f"  - CTC weight: {config['training']['ctc_weight']} (was 0.3)")
print(f"  - Learning rate: {config['training']['learning_rate']} (was 0.0001)")
print(f"  - Gradient clip: {config['training']['grad_clip']} (was None)")
print(f"  - Testing: Every {config['training']['test_every']} epochs")

## Step 6: Start Training with Periodic Testing

In [ ]:
# Start training
!python /kaggle/working/training_scripts/train_konkanivani_asr.py \
    --config /kaggle/working/config/training_config_fixed.yaml \
    --epochs 100

## Step 7: Monitor Progress

### Expected Timeline:
- **Epoch 1-10**: Blank prob 95-98% (learning basics)
- **Epoch 10-20**: Blank prob 80-90% (characters appearing)
- **Epoch 20-40**: Blank prob 50-80% ✅ **WORKING!**
- **Epoch 40-100**: Blank prob 30-50% (refinement)

In [ ]:
# Check test results
test_results_dir = Path('/kaggle/working/checkpoints')
test_files = sorted(test_results_dir.glob('test_results_epoch_*.json'))

if test_files:
    print("Test Results Summary:")
    print("=" * 80)
    for test_file in test_files:
        with open(test_file) as f:
            results = json.load(f)
            epoch = results.get('epoch', '?')
            blank_prob = results.get('avg_blank_prob', 0)
            status = '✅ WORKING!' if blank_prob < 80 else '❌ Not yet'
            print(f"Epoch {epoch:3d}: Blank prob {blank_prob:5.1f}% - {status}")
else:
    print("No test results yet. Check back after epoch 5.")

## Step 8: Download Best Checkpoint

In [ ]:
# Find best checkpoint (lowest validation loss)
checkpoint_dir = Path('/kaggle/working/checkpoints')
checkpoints = sorted(checkpoint_dir.glob('checkpoint_epoch_*.pt'))

if checkpoints:
    best_ckpt = None
    best_val_loss = float('inf')
    
    for ckpt_path in checkpoints:
        ckpt = torch.load(ckpt_path, map_location='cpu')
        val_loss = ckpt.get('val_loss', float('inf'))
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_ckpt = ckpt_path
    
    print(f"Best checkpoint: {best_ckpt.name}")
    print(f"Validation loss: {best_val_loss:.4f}")
    
    # Copy to best_model.pt
    import shutil
    shutil.copy(best_ckpt, checkpoint_dir / 'best_model.pt')
    print("✓ Saved as best_model.pt")
else:
    print("No checkpoints found yet")

In [ ]:
# Create download link
from IPython.display import FileLink

print("Download your trained model:")
FileLink('/kaggle/working/checkpoints/best_model.pt')

## Step 9: Quick Test

In [ ]:
# Test the best model on a few samples
!python /kaggle/working/scripts/test_best_model.py \
    --checkpoint /kaggle/working/checkpoints/best_model.pt \
    --max_files 10

## Summary

### Key Fixes Applied:
1. ✅ CTC weight: 0.3 → 0.8 (critical for transcription)
2. ✅ Learning rate: 0.0001 → 0.0003 (faster learning)
3. ✅ Added gradient clipping: 5.0 (stability)
4. ✅ Full dataset: 21h → 88h (4x more data)
5. ✅ Periodic testing: Monitor every 5 epochs

### Expected Results:
- Model should start working by epoch 20-30
- Blank probability should drop below 80%
- Transcriptions should be recognizable
- Final CER should be 20-40%

### Next Steps:
1. Download best_model.pt
2. Test locally on your audio files
3. Deploy for production use